# exp125_confidence_gate_continuity_rawtest_parity train

Posthoc train-side audit for exp102/exp112 confidence gates. This notebook does not train a new model and does not create a submission candidate.

## Contents

1. Setup and configuration
2. Input and variant contract
3. Run continuity and parity audit
4. Metrics and generated artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from confidence_gate_continuity_rawtest_parity import run_audit

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_ROWS_ENV = os.environ.get("EXPERIMENT_MAX_ROWS_PER_VARIANT")
MAX_ROWS_PER_VARIANT = int(MAX_ROWS_ENV) if MAX_ROWS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Output root:", paths.output_root)
print("Artifacts:", paths.artifacts_dir)
print("Debug:", DEBUG, "Max rows per variant:", MAX_ROWS_PER_VARIANT)

## 2. Input and variant contract

In [ ]:
contract = {
    "exp102_variants": get_nested(config, "audit.exp102_variants"),
    "exp112_variants": get_nested(config, "audit.exp112_variants"),
    "required_shared_variants": get_nested(config, "audit.required_shared_variants"),
    "surface_baseline_variants": get_nested(config, "audit.surface_baseline_variants"),
    "guardrails": get_nested(config, "audit.guardrails"),
}
print(json.dumps(contract, indent=2, ensure_ascii=False))

## 3. Run continuity and parity audit

In [ ]:
summary = run_audit(config, paths, max_rows_per_variant=MAX_ROWS_PER_VARIANT)
paths.metrics_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True) + "\n")
print(json.dumps({
    "status": summary["status"],
    "fair_shared_surface": summary["fair_shared_surface"],
    "best_fair_shared_variant": summary["best_fair_shared_variant"],
    "guardrails": summary["guardrails"],
    "decision": summary["decision"],
}, indent=2, ensure_ascii=False))
print("Metrics written:", paths.metrics_path)

## 4. Metrics and generated artifacts

In [ ]:
artifact_paths = {key: Path(value) for key, value in summary["artifacts"].items() if key != "summary"}
display(pd.DataFrame([{"artifact": key, "path": str(path), "exists": path.exists()} for key, path in artifact_paths.items()]))

metrics = pd.read_csv(artifact_paths["metrics"])
display(metrics.sort_values(["surface_scope", "rmse_tvt"]).head(20))

continuity = pd.read_csv(artifact_paths["continuity_summary"])
display(continuity.sort_values(["surface", "variant"]))

parity = pd.read_csv(artifact_paths["rawtest_parity_checklist"])
display(parity)